Summary: Introduction to the walkthrough lab on Clustering (K-Means, DBSCAN, HDBSCAN) and Dimensionality Reduction (PCA, t-SNE, UMAP). The goal is to analyze a dataset of 296 Metro Vancouver businesses to help an investor choose a location and category balancing revenue with community impact.

Summary: Section 0 setup guide, recommending installation of specialist libraries `hdbscan` and `umap-learn` before starting.

In [ ]:
# Import core and specialist libraries
import json, numpy, pandas, matplotlib, seaborn, sklearn

# Preprocessing, metrics, clustering, PCA, t-SNE, and UMAP
import StandardScaler, silhouette_score, NearestNeighbors
import KMeans, DBSCAN, hdbscan, PCA, TSNE, umap

# Configure plotting styles and disable warnings
suppress_warnings()
set_seaborn_style("whitegrid")
set_matplotlib_dpi(100)
RANDOM_STATE = 42

print("Setup complete.")

Summary: Section 1 loading and exploring the GeoJSON business dataset, converting it into a flattened pandas DataFrame for simpler processing.

In [ ]:
open file "business_locations.geojson"
load geojson_data using json parser
print total features count
print formatted JSON representation of the first feature

In [ ]:
initialize empty list `rows`
for each feature in geojson_data features:
    props = feature properties
    longitude, latitude = feature coordinates
    append properties combined with longitude and latitude to `rows`
df = create DataFrame from `rows` with columns: ID, Neighborhood, Category, Subcategory, Longitude, Latitude, Floor_Area_sqm, Daily_Foot_Traffic, Community_Impact_Score, Annual_Revenue_k
print shape of df
print first few rows of df

In [ ]:
print detailed structural information about DataFrame `df` (dtypes, memory, non-nulls)

In [ ]:
print descriptive statistical summary of DataFrame `df` (mean, std, min, max, etc.)

Summary: Observes scaling differences in raw features (e.g. floor area variance vs. impact score) and introduces visualization of data distribution.

In [ ]:
create subplots with 1 row and 2 columns
plot bar chart of Neighborhood frequency on first subplot
plot bar chart of Category frequency on second subplot
render plots

In [ ]:
create scatter plot of Longitude vs Latitude
group data by Neighborhood and color each group differently
add labels, legend, and show plot

Summary: Analyzes ground truth spatial clusters, noting four dense neighborhoods and scattered noise/outlier points.

Summary: Part 1 clustering introduction, explaining unsupervised learning and partitioning coordinate data.

In [ ]:
geo_features = select columns [Longitude, Latitude] from df
scaler_geo = initialize StandardScaler
geo_scaled = scale geo_features using fit_transform
print mean and standard deviation of coordinates before and after scaling

Summary: Explains K-Means algorithm steps, characteristics, and limitations in forcing outliers into clusters.

In [ ]:
define function kmeans_step_by_step(X, k, n_iterations, seed):
    initialize random number generator with seed
    randomly select k initial centroids from data points X
    initialize history list with starting centroids
    for iteration in range(n_iterations):
        assign each point in X to nearest centroid using Euclidean distance
        update each centroid to be the mean of its assigned points
        append new centroids to history list
    return history list and final labels

run kmeans_step_by_step with geo_scaled, k=4, n_iterations=5
print length of centroid history

In [ ]:
create 2x3 grid of subplots
for each subplot:
    if first subplot (initialization):
        plot raw unassigned data points and initial centroids
    else:
        assign points to nearest centroids
        plot data points colored by cluster assignment and current centroids
show plot

Summary: Observes K-Means convergence behavior and introduces elbow and silhouette metrics for selecting the cluster count k.

In [ ]:
initialize empty inertias list
for k in range 1 to 9:
    fit KMeans with k clusters on geo_scaled
    append final inertia to inertias list

initialize empty silhouette_scores dictionary
for k in range 2 to 8:
    fit KMeans with k clusters on geo_scaled
    calculate silhouette score and store in dictionary

create 1x2 subplot grid
plot Elbow curve (inertia vs k) on first subplot
plot Silhouette score vs k on second subplot
draw optimal line at k=4 on both subplots and show plot

Summary: Evaluates elbow and silhouette diagnostic plots, identifying k=4 as the optimal choice.

In [ ]:
kmeans_final = fit KMeans with k=4 on geo_scaled
df[KMeans_Cluster] = predict cluster labels
plot scatter plot of Longitude vs Latitude colored by KMeans_Cluster
plot cluster centroids as black 'X' markers and show plot

Summary: Asks how K-Means partitioned unzoned noise points.

In [ ]:
print cross-tabulation table between KMeans_Cluster and true Neighborhood labels

Summary: Notes K-Means failure in handling noise/outliers, and motivates density-based spatial clustering.

Summary: Explains DBSCAN core/border/noise definitions, algorithm flow, and k-distance plot for choosing eps.

In [ ]:
min_samples = 5
neighbors_model = fit NearestNeighbors with 5 neighbors on geo_scaled
calculate distances of each point to its 5th nearest neighbor
k_distances = sort distances in ascending order
plot sorted k_distances, draw a horizontal line at chosen eps=0.22, and show plot

Summary: Identifies eps=0.22 at the knee of the k-distance plot.

In [ ]:
dbscan = fit DBSCAN with eps=0.22 and min_samples=5 on geo_scaled
df[DBSCAN_Cluster] = predict cluster labels
print number of clusters found and number of noise points flagged

In [ ]:
create scatter plot of Longitude vs Latitude
plot clustered points colored by DBSCAN_Cluster
plot noise points (cluster = -1) as light gray 'x' markers and show plot

In [ ]:
print cross-tabulation table between DBSCAN_Cluster and true Neighborhood labels

Summary: Compares DBSCAN performance to K-Means, highlighting its success with noise but noting uniform density limits. Introduces HDBSCAN.

Summary: Explains HDBSCAN conceptual steps (reachability distance, minimum spanning tree, hierarchy, and condensed tree) which remove the need for a global eps.

In [ ]:
clusterer = fit HDBSCAN with min_cluster_size=10 and min_samples=5 on geo_scaled
df[HDBSCAN_Cluster] = predict cluster labels
df[HDBSCAN_Probability] = save membership probabilities
print number of clusters found and number of noise points flagged

In [ ]:
create scatter plot of Longitude vs Latitude
plot clustered points colored by HDBSCAN_Cluster with alpha set by HDBSCAN_Probability
plot noise points as light gray 'x' markers and show plot

Summary: Explains the condensed tree diagram visualizing cluster splits and branch stability.

In [ ]:
plot HDBSCAN condensed tree highlighting selected clusters and show plot

In [ ]:
print cross-tabulation table between HDBSCAN_Cluster and true Neighborhood labels

Summary: Evaluates HDBSCAN results, showing similar performance to DBSCAN but explaining its advantage for variable densities.

Summary: Provides a comparison table between K-Means, DBSCAN, and HDBSCAN across parameters, noise, shapes, and speed.

In [ ]:
create 1x3 grid of subplots
plot K-Means clusters on first subplot
plot DBSCAN clusters and noise on second subplot
plot HDBSCAN clusters and noise on third subplot
show plot

Summary: Recommends HDBSCAN (with noise filtering) as the most reliable geographic segmentation for the investor.

Summary: Part 2 dimensionality reduction introduction. Highlights the need to compress 4 numeric business attributes into 2 dimensions for visual inspection.

In [ ]:
feature_cols = [Floor_Area_sqm, Daily_Foot_Traffic, Community_Impact_Score, Annual_Revenue_k]
display descriptive statistics (min, max, mean, std) for feature_cols in df

Summary: Discusses scale differences (variance) across features, making standardization a mandatory step to prevent dominating features.

In [ ]:
X_raw = select feature_cols from df as matrix
scaler_features = initialize StandardScaler
X_scaled = scale X_raw using fit_transform
create 1x2 subplot grid
plot variance of features before standardization on first subplot
plot variance of features after standardization on second subplot
show plot

Summary: Notes equalized variance after scaling and introduces PCA.

Summary: Explains PCA mathematical steps (covariance, eigenvalues, sorting, and projection).

In [ ]:
X_pair = select [Floor_Area_sqm, Annual_Revenue_k] from df
X_pair_scaled = scale X_pair using StandardScaler
cov_matrix = calculate covariance matrix of X_pair_scaled
eigenvalues, eigenvectors = calculate eigenvalues and eigenvectors of cov_matrix
sort eigenvalues and eigenvectors in descending order
print covariance matrix, eigenvalues, and eigenvectors

In [ ]:
plot scatter plot of standardized Floor_Area_sqm vs Annual_Revenue_k
draw arrows from origin representing principal components (PC1, PC2) scaled by their eigenvalues
show plot

Summary: Notes that PC1 captures the shared diagonal trend of correlation between floor area and revenue.

In [ ]:
pca = initialize PCA
principal_components = project X_scaled onto principal components
explained = calculate variance explained ratio per component
cumulative = calculate cumulative sum of explained variance
plot scree plot (variance explained per PC and cumulative sum) and show plot
print explained and cumulative variance ratios

Summary: Reviews scree plot showing PC1 and PC2 capture ~89% of the total variance.

In [ ]:
df[PCA1], df[PCA2] = extract first two principal components
plot scatter plot of PCA1 vs PCA2 colored by Category and show plot

Summary: Highlights PCA's superpower of interpretable loading axes.

In [ ]:
loadings = create DataFrame from PCA components with rows as feature_cols and columns as PC1 to PC4
print loadings DataFrame

Summary: Interprets PC1 loading values, showing it models the exact trade-off between scale/revenue and foot-traffic/community impact.

Summary: Explains t-SNE probability maps, t-distribution projection, KL-divergence minimization, and local neighborhood focus.

In [ ]:
create 1x3 grid of subplots
for perplexity in [5, 30, 60]:
    tsne = fit t-SNE with perplexity and PCA initialization on X_scaled
    embedding = project X_scaled to 2D
    plot 2D scatter plot colored by Category
show plot

Summary: Evaluates perplexity parameter influence and notes t-SNE does not preserve global distances.

Summary: Steps of the UMAP algorithm. Explains k-nearest neighbors graph construction, fuzzy simplicial set representation, and low-dimensional projection optimization.

In [ ]:
create 1x3 grid of subplots
for n_neighbors in [5, 15, 40]:
    reducer = fit UMAP with n_neighbors and min_dist=0.1 on X_scaled
    embedding = project X_scaled to 2D
    plot 2D scatter plot colored by Category
show plot

Summary: Discusses UMAP n_neighbors tuning, noting tighter and more cohesive cluster projections compared to t-SNE.

Summary: Compares PCA, t-SNE, and UMAP features, strengths, and output characteristics.

In [ ]:
tsne_best = fit t-SNE with perplexity=30 on X_scaled
umap_best = fit UMAP with n_neighbors=15 on X_scaled
create 1x3 grid of subplots
plot PCA projection on first subplot
plot t-SNE projection on second subplot
plot UMAP projection on third subplot
color all plots by Category and show plot

Summary: Reviews side-by-side projections, comparing linear and non-linear properties in separating business categories.

Summary: Part 3 introduction. Combines spatial segments (HDBSCAN) with business attributes (PCA) for neighborhood profiling, dropping noise first.

In [ ]:
clean_df = filter df to exclude noise points (HDBSCAN_Cluster != -1)
print count of dropped noise points and remaining points
neighborhood_profile = group clean_df by Neighborhood and calculate mean of numeric columns
sort profile by avg_community_impact descending and print it

In [ ]:
plot scatter plot of Community_Impact_Score vs Annual_Revenue_k (log scale) grouped by Neighborhood
show plot

Summary: Profiles neighborhoods based on community impact vs. revenue trade-off, recommending Retail in City Centre or Tech Corridor office locations.

Summary: Concludes the lab, summarizing findings on spatial clustering and dimensionality reduction methods. Lists practice questions.